In [1]:
import pandas as pd


In [2]:
!pip install -q ydata-profiling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 2.5 MB/s eta 0:00:00


In [3]:
data = pd.read_csv("/content/drive/MyDrive/HQMTV PROMPT STUDIO/insurance.csv")

In [4]:
data

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [5]:
from ydata_profiling import ProfileReport
from google.colab import files

# Generate profiling report
profile = ProfileReport(
    data,
    title=" HQ MTV Dataset Profiling report",
    explorative=True
)

# Save report as HTML
report_file = "HQ MTV data_profiling report.html"
profile.to_file(report_file)

print(f"Report saved as: {report_file}")

# Download the HTML report
files.download(report_file)

/tmp/ipykernel_1180/618691412.py:1: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 7/7 [00:00<00:00, 38.81it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Report saved as: HQ MTV data_profiling report.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
from sklearn.model_selection import train_test_split

X = data.drop("charges", axis=1)
y = data["charges"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = ["sex", "smoker", "region"]
numerical_features = ["age", "bmi", "children"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge())
])

ridge_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'bmi', 'children']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['sex', 'smoker',
                                                   'region'])])),
                ('model', Ridge())])

In [10]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])

rf_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'bmi', 'children']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['sex', 'smoker',
                                                   'region'])])),
                ('model', RandomForestRegressor(random_state=42))])

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 3, 5],
    "model__criterion": ["squared_error"]
}

grid = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=5,
    scoring="r2"
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(grid.best_params_)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.7)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red",
    linewidth=2
)

plt.xlabel("Actual Charges")
plt.ylabel("Predicted Charges")
plt.title("Predicted vs Actual Insurance Charges")

plt.grid(True)
plt.show()

In [ ]:
import joblib

joblib.dump(best_model, "insurance_model.pkl")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))

plt.scatter(y_test, y_pred, alpha=0.7)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red",
    linewidth=2
)

plt.xlabel("Actual Insurance Charges")
plt.ylabel("Predicted Insurance Charges")
plt.title("Predicted vs Actual Insurance Charges")
plt.grid(True)

# Save the figure
plt.savefig("Figure_5_Predicted_vs_Actual.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
import joblib

joblib.dump(best_model, "insurance_model.pkl")
print("Model saved successfully!")

In [ ]:
%%writefile app.py

import streamlit as st
import joblib
import pandas as pd

# Load trained model
model = joblib.load("insurance_model.pkl")

st.title("Medical Insurance Charges Predictor")

age = st.number_input("Age", min_value=18, max_value=100, value=30)
sex = st.selectbox("Sex", ["male", "female"])
bmi = st.number_input("BMI", min_value=10.0, max_value=60.0, value=25.0)
children = st.number_input("Children", min_value=0, max_value=10, value=0)
smoker = st.selectbox("Smoker", ["yes", "no"])
region = st.selectbox(
    "Region",
    ["northeast", "northwest", "southeast", "southwest"]
)

if st.button("Predict Insurance Charges"):
    input_data = pd.DataFrame({
        "age": [age],
        "sex": [sex],
        "bmi": [bmi],
        "children": [children],
        "smoker": [smoker],
        "region": [region]
    })

    prediction = model.predict(input_data)

    st.success(f"Predicted Insurance Charges: ${prediction[0]:,.2f}")

In [ ]:
!pip install -q streamlit pyngrok

In [ ]:
!streamlit run app.py &



2026-07-30 20:27:30.711 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.74.233.137:8501

  Stopping...


In [ ]:
import joblib

joblib.dump(best_model, "insurance_model.pkl")

['insurance_model.pkl']

In [ ]:
!pip install streamlit

In [ ]:
!pip install -q gradio

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

rf_default = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])

rf_default.fit(X_train, y_train)

y_pred_default = rf_default.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predictions
y_pred_default = rf_default.predict(X_test)

# Metrics
r2 = r2_score(y_test, y_pred_default)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_default))
mae = mean_absolute_error(y_test, y_pred_default)

print(f"R² = {r2:.2f}")
print(f"RMSE = ${rmse:.2f}")
print(f"MAE = ${mae:.2f}")

R² = 0.86
RMSE = $4582.97
MAE = $2541.61


In [ ]:
# rf_default.fit(X_train_encoded, y_train)
# y_pred_default = rf_default.predict(X_test_encoded)

NameError: name 'X_train_encoded' is not defined

In [ ]:
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# import numpy as np

# rf_default = RandomForestRegressor(random_state=42)

# rf_default.fit(X_train, y_train)

# y_pred_default = rf_default.predict(X_test)

# r2 = r2_score(y_test, y_pred_default)
# rmse = np.sqrt(mean_squared_error(y_test, y_pred_default))
# mae = mean_absolute_error(y_test, y_pred_default)

# print("R² =", round(r2, 2))
# print("RMSE =", round(rmse, 2))
# print("MAE =", round(mae, 2))

ValueError: could not convert string to float: 'female'

In [ ]:
import gradio as gr
import joblib
import pandas as pd

model = joblib.load("insurance_model.pkl")

def predict(age, sex, bmi, children, smoker, region):
    df = pd.DataFrame({
        "age": [age],
        "sex": [sex],
        "bmi": [bmi],
        "children": [children],
        "smoker": [smoker],
        "region": [region]
    })
    return float(model.predict(df)[0])

demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Number(label="Age"),
        gr.Dropdown(["male", "female"], label="Sex"),
        gr.Number(label="BMI"),
        gr.Number(label="Children"),
        gr.Dropdown(["yes", "no"], label="Smoker"),
        gr.Dropdown(["northeast", "northwest", "southeast", "southwest"], label="Region"),
    ],
    outputs="number",
    title="Medical Insurance Charges Predictor"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bafa73238915e409e8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
%%writefile plot_charges_app.py

import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(layout="wide")

st.title("Insurance Charges Distribution")

# Load the dataset (assuming it's in the same location as in the notebook)
data = pd.read_csv("/content/drive/MyDrive/HQMTV PROMPT STUDIO/insurance.csv")

# Create a histogram of the 'charges' column
fig = px.histogram(data, x="charges", nbins=50, title="Distribution of Insurance Charges")
fig.update_layout(bargap=0.1) # Add some gap between bars
st.plotly_chart(fig, use_container_width=True)

Writing plot_charges_app.py


In [ ]:
# Give Streamlit a moment to start and then get the public URL
import time
from pyngrok import ngrok

time.sleep(5) # Wait for Streamlit to start

# Disconnect any previous ngrok tunnels
ngrok.kill()

# Establish a new ngrok tunnel to the Streamlit app port (8501)
public_url = ngrok.connect(8501)
print(f"Your Streamlit app is live at: {public_url}")